In [1]:
# # Set up for running selenium in Google Colab
# %%shell
# sudo apt -y update
# sudo apt install -y wget curl unzip
# wget http://archive.ubuntu.com/ubuntu/pool/main/libu/libu2f-host/libu2f-udev_1.1.4-1_all.deb
# dpkg -i libu2f-udev_1.1.4-1_all.deb
# wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# dpkg -i google-chrome-stable_current_amd64.deb
# CHROME_DRIVER_VERSION=`curl -sS chromedriver.storage.googleapis.com/LATEST_RELEASE`
# wget -N https://chromedriver.storage.googleapis.com/$CHROME_DRIVER_VERSION/chromedriver_linux64.zip -P /tmp/
# unzip -o /tmp/chromedriver_linux64.zip -d /tmp/
# chmod +x /tmp/chromedriver
# mv /tmp/chromedriver /usr/local/bin/chromedriver
# pip install selenium
# pip install chromedriver-autoinstaller-fix

In [2]:
import chromedriver_autoinstaller_fix
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import time
import os
import shutil
import pandas as pd
import requests

In [3]:
PROXY_HOSTS=['154.6.99.159','38.62.221.210','154.6.99.65',
             '38.62.223.107','38.62.222.126','38.62.223.126',
             '38.62.223.53','154.6.96.179','38.62.223.173',
             '38.62.222.114','154.6.97.13','154.6.98.73','154.6.98.40',
             '154.6.96.44','38.62.220.93','154.6.99.194','38.62.221.13',
             '154.6.96.60','154.6.99.162','154.6.99.93','154.6.98.126',
             '38.62.221.49','38.62.220.172','154.6.96.26','38.62.221.2']

PROXY_PORT = 3128

current_try = 0

# Function to initialize the WebDriver
def initialize_driver(download_folder = r"D:\learning\thesis"):
    global current_try
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--ignore-ssl-errors=yes')
    chrome_options.add_argument('--ignore-certificate-errors')
    proxy_host = PROXY_HOSTS[current_try%10]
    chrome_options.add_argument(f'--proxy-server=http://{proxy_host}:{PROXY_PORT}')
    chrome_options.add_experimental_option('excludeSwitches', ['enable-logging'])
    chrome_options.add_experimental_option("prefs", {
      "download.default_directory": download_folder
      })
    
    driver = webdriver.Chrome(options = chrome_options)
    current_try += 1
    
    try:
        driver.get("https://www.kaggle.com/")
        if(driver.find_elements("xpath","//*[contains(text(), 'This page isn’t working')]")):
            driver.quit()
            return initialize_driver()
        return driver
    except Exception as e:
        print(f"Exception: {e}")
        driver.quit()
        return initialize_driver()

In [4]:
def get_notebook_labels(notebook_url):
    # Navigate to the notebook URL
    driver.get(notebook_url)
    
    # Initialize labels to None
    labels = None
    
    # Check if the 'Python' label exists on the page
    if len(driver.find_elements("xpath", "//*[contains(text(), 'Python')]")) > 0:
        # Get the first element that contains the text 'Python'
        labels_row = driver.find_elements("xpath", "//*[contains(text(), 'Python')]")[0]

        # Find each label link within the labels row
        labels = [label.get_attribute('innerHTML') for label in labels_row.find_elements(By.TAG_NAME, 'a')]

    # Print the labels for the given notebook URL
    print("Labels at URL %s are %s" % (notebook_url, labels))
    
    # Return the list of labels
    return labels

In [5]:
# Function to scrape the Kaggle Rankings table
def scrape_kaggle_notebooks_per_user(username, display_name, user_gender, download_folder = r"D:\learning\thesis"):

    user_code_url = f'https://www.kaggle.com/{username}/code'
    
    driver = initialize_driver()
    driver.get(user_code_url)
    
    # hide cookies divs
    driver.execute_script("document.getElementsByClassName('sc-eoeFZW')[0].style.display = 'none';")
    
    # Wait for 10 seconds to allow the page to load completely
    time.sleep(10)

    if(driver.find_elements(By.CLASS_NAME, 'km-list--avatar-list')):

        # Get the initial number of items in the list
        final_item_count = len(driver.find_element(By.CLASS_NAME, 'km-list--avatar-list').find_elements(By.TAG_NAME, 'li'))

        # Set the time limit for scrolling (30 minutes)
        timeout = time.time() + 30 * 60

        while time.time() < timeout:

            # Extract data from the table rows
            list_items = driver.find_element(By.CLASS_NAME, 'km-list--avatar-list').find_elements(By.TAG_NAME, 'li')

            # Scroll to the bottom of the div
            driver.execute_script("arguments[0].scrollIntoView();", driver.find_element(By.CLASS_NAME, 'km-list--avatar-list').find_elements(By.TAG_NAME, 'li')[-1])

            # Wait for a short time to allow new content to load (adjust sleep time if needed)
            time.sleep(2)

            new_item_count = len(driver.find_element(By.CLASS_NAME, 'km-list--avatar-list').find_elements(By.TAG_NAME, 'li'))

            # If no new items are added, break the loop
            if new_item_count == final_item_count:
                break
            else:
                final_item_count = new_item_count

        i=0
        while i < len(list_items):
            
            try:
                
                current_item = driver.find_element(By.CLASS_NAME, 'km-list--avatar-list').find_elements(By.TAG_NAME, 'li')[i]
                
                # Find the menu button by its tag
                menu_element = current_item.find_element(By.CLASS_NAME,'mdc-menu-surface--anchor')

                # Find the notebook url of the current list item
                notebook_url = current_item.find_elements(By.CLASS_NAME, 'sc-jegxcv')[-1].get_attribute('href')[:-8]

                # #Simulate a click on the button
                ActionChains(driver).move_to_element(menu_element).click(menu_element).perform()

                # Wait for 3 seconds to allow the card to open
                time.sleep(3)

                # Find the download code button by its class name
                actions_card = driver.find_elements(By.CLASS_NAME, 'mdc-menu-surface--open')[-1]

                download_button = actions_card.find_element(By.TAG_NAME,'a')

                # Simulate a click on the button
                download_button.click()

                print("Download initiated successfully : ",notebook_url)

                # Wait for 5 seconds to allow the ipynp to download completely
                time.sleep(5)

                # Specify path
                path = f'{download_folder}\\{notebook_url.split("/")[-2]}.ipynb'


                # Check whether the specified
                # path exists or not
                if(os.path.exists(path)):

                    new_path = f'{download_folder}\\{user_gender}\\{username}'
                    if not os.path.exists(new_path):
                        os.makedirs(new_path)
                    shutil.move(path,new_path)

                    users_code_location_df.loc[len(users_code_location_df.index)] = [
                        username, display_name, user_gender, notebook_url, f'{new_path}/{notebook_url.split("/")[-2]}.ipynb', None]

                    print("Download finished successfully.")
                else:
                    print("An error occurred.")
            except:
                print("stopped due to Too many requests switching to new IP")
                
                # Close the webdriver
                driver.quit()
                
                driver = initialize_driver()
                driver.get(user_code_url)
                
                # hide cookies div
                driver.execute_script("document.getElementsByClassName('sc-eoeFZW')[0].style.display = 'none';")
                
                # Get the initial number of items in the list
                final_item_count = len(driver.find_element(By.CLASS_NAME, 'km-list--avatar-list').find_elements(By.TAG_NAME, 'li'))

                # Set the time limit for scrolling (30 minutes)
                timeout = time.time() + 30 * 60

                while time.time() < timeout:

                    # Extract data from the table rows
                    list_items = driver.find_element(By.CLASS_NAME, 'km-list--avatar-list').find_elements(By.TAG_NAME, 'li')

                    # Scroll to the bottom of the div
                    driver.execute_script("arguments[0].scrollIntoView();", list_items[-1])

                    # Wait for a short time to allow new content to load (adjust sleep time if needed)
                    time.sleep(2)

                    new_item_count = len(driver.find_element(By.CLASS_NAME, 'km-list--avatar-list').find_elements(By.TAG_NAME, 'li'))

                    # If no new items are added, break the loop
                    if new_item_count == final_item_count:
                        break
                    else:
                        final_item_count = new_item_count
                
                current_item = driver.find_element(By.CLASS_NAME, 'km-list--avatar-list').find_elements(By.TAG_NAME, 'li')[i]
                # Find the menu button by its tag
                menu_element = current_item.find_element(By.CLASS_NAME,'mdc-menu-surface--anchor')

                # Find the notebook url of the current list item
                notebook_url = current_item.find_elements(By.CLASS_NAME, 'sc-jegxcv')[-1].get_attribute('href')[:-8]

                # #Simulate a click on the button
                ActionChains(driver).move_to_element(menu_element).click(menu_element).perform()

                # Wait for 3 seconds to allow the card to open
                time.sleep(3)

                # Find the download code button by its class name
                actions_card = driver.find_elements(By.CLASS_NAME, 'mdc-menu-surface--open')[-1]

                download_button = actions_card.find_element(By.TAG_NAME,'a')

                # Simulate a click on the button
                download_button.click()

                print("Download initiated successfully : ",notebook_url)

                # Wait for 5 seconds to allow the ipynp to download completely
                time.sleep(5)

                # Specify path
                path = f'{download_folder}\\{notebook_url.split("/")[-2]}.ipynb'
                
                # Check whether the specified
                # path exists or not
                if(os.path.exists(path)):

                    new_path = f'{download_folder}\\{user_gender}\\{username}'
                    if not os.path.exists(new_path):
                        os.makedirs(new_path)
                    shutil.move(path,new_path)

                    users_code_location_df.loc[len(users_code_location_df.index)] = [
                        username, display_name, user_gender, notebook_url, f'{new_path}/{notebook_url.split("/")[-2]}.ipynb', None]

                    print("Download finished successfully.")
                else:
                    print("An error occurred.")
            i+=1
    else:
        print("This user does not have a code section",username)


    # Close the webdriver
    driver.quit()

In [6]:
users_with_gender = pd.read_csv('users_with_genders_prediction.csv', index_col=0)

users_code_location_df = pd.DataFrame(columns = ['Username', 'Display_Name','Gender', 'notebook_url',"code_location","labels"]) 

In [7]:
# Filter out rows where the gender prediction is 'none'
users_with_gender = users_with_gender[users_with_gender["First_Name_Guess-genderize"] != 'none'].reset_index(drop=True)

users_with_gender

,Username,Display_Name,Image_Path,First_Name,First_Name-genderize_results,First_Name_Guess-genderize
0,christofhenkel,Dieter,"""https://storage.googleapis.com/kaggle-avatars...",Dieter,"{'count': 40603, 'name': 'Dieter', 'gender': '...",male
1,cdeotte,Chris Deotte,"""https://storage.googleapis.com/kaggle-avatars...",Chris,"{'count': 256938, 'name': 'Chris', 'gender': '...",male
2,ilu000,Pascal Pfeiffer,"""https://storage.googleapis.com/kaggle-avatars...",Pascal,"{'count': 199383, 'name': 'Pascal', 'gender': ...",male
3,aerdem4,Ahmet Erdem,"""https://storage.googleapis.com/kaggle-avatars...",Ahmet,"{'count': 288615, 'name': 'Ahmet', 'gender': '...",male
4,vladimirsydor,Volodymyr,"""https://storage.googleapis.com/kaggle-avatars...",Volodymyr,"{'count': 1017, 'name': 'Volodymyr', 'gender':...",male
...,...,...,...,...,...,...
6246,iguyon,Isabelle,"""https://storage.googleapis.com/kaggle-avatars...",Isabelle,"{'count': 245637, 'name': 'Isabelle', 'gender'...",female
6247,chrisraimondi,Chris Raimondi,"""https://storage.googleapis.com/kaggle-avatars...",Chris,"{'count': 256938, 'name': 'Chris', 'gender': '...",male
6248,miroslaw,Miroslaw Horbal,"""https://storage.googleapis.com/kaggle-avatars...",Miroslaw,"{'count': 4469, 'name': 'Miroslaw', 'gender': ...",male
6249,adamwoz,Adam,"""https://storage.googleapis.com/kaggle-avatars...",Adam,"{'count': 554061, 'name': 'Adam', 'gender': 'm...",male


In [8]:
for index, row in users_with_gender.iterrows():
    print("Started scraping at row number %s for user %s"%(index, row["Username"]))
    scrape_kaggle_notebooks_per_user(row["Username"],row["Display_Name"],row["First_Name_Guess-genderize"])
    print("Finished scraping at row number ",index)

In [13]:
# Initiallize the webdriver
driver = initialize_driver()

for index in range(len(users_code_location_df.index)):
    users_code_location_df.at[index , 'labels'] = get_notebook_labels(users_code_location_df.iloc[index].notebook_url)
    print("finish running till %s"%(index))
        
# Close the webdriver
driver.quit();

labels at url https://www.kaggle.com/code/tunguz/covid-19-w3-a-few-charts-and-a-simple-baseline are ['Covid19 Metadata', 'COVID19 Global Forecasting (Week 3)']
finish running till 58276
labels at url https://www.kaggle.com/code/tunguz/covid-19-week-3-blend-2 are ['SmokingStats', 'Covid19 Metadata', 'countryinfo']
finish running till 58277
labels at url https://www.kaggle.com/code/tunguz/covid-19-week-4 are ['Covid19 Forecasting Metadata', 'COVID19 Global Forecasting (Week 4)']
finish running till 58278
labels at url https://www.kaggle.com/code/tunguz/covid-19-week-5-using-xgboost are ['COVID19 Global Forecasting (Week 5)']
finish running till 58279
labels at url https://www.kaggle.com/code/tunguz/covid-daily-rate-starter-notebook are ['Data on COVID-19 (coronavirus) ']
finish running till 58280
labels at url https://www.kaggle.com/code/tunguz/covid-vaccination-eda-starter are ['The Our World in Data COVID Vaccination Data']
finish running till 58281
labels at url https://www.kaggle.com

labels at url https://www.kaggle.com/code/tunguz/google-trends-2020-visualization are None
finish running till 58325
labels at url https://www.kaggle.com/code/tunguz/google-trends-2021-visualization are None
finish running till 58326
labels at url https://www.kaggle.com/code/tunguz/gpu-shap-values-with-mercedes-data are ['Mercedes-Benz Greener Manufacturing']
finish running till 58327
labels at url https://www.kaggle.com/code/tunguz/gpu-xgb-cats-vs-dogs-with-irv2 are ['Cats and Dogs Embedded Data', 'Dogs vs. Cats Redux: Kernels Edition']
finish running till 58328
labels at url https://www.kaggle.com/code/tunguz/grains-eda are ['Feed Grains: Yearbook Tables']
finish running till 58329
labels at url https://www.kaggle.com/code/tunguz/gutenberg-placeholder are ['The PG-19 Language Modeling Benchmark Dataset']
finish running till 58330
labels at url https://www.kaggle.com/code/tunguz/h2o-automl-aggregator-and-submission are ['TReNDS Neuroimaging', 'TReNDS Train Test Creator', 'TReNDS H2O A

labels at url https://www.kaggle.com/code/tunguz/last-place-solution-tps-07-2021 are ['Tabular Playground Series - Jul 2021']
finish running till 58381
labels at url https://www.kaggle.com/code/tunguz/lgbm-gpu-hyperparameters-with-optuna-dummies are ['Tabular Playground Series - Feb 2021']
finish running till 58382
labels at url https://www.kaggle.com/code/tunguz/lightgbm-baseline are ['Kannada MNIST']
finish running till 58383
labels at url https://www.kaggle.com/code/tunguz/logistic-regression-0 are None
finish running till 58384
labels at url https://www.kaggle.com/code/tunguz/logistic-regression-tfidf are None
finish running till 58385
labels at url https://www.kaggle.com/code/tunguz/logistic-regression-with-tf-embeddings are None
finish running till 58386
labels at url https://www.kaggle.com/code/tunguz/lr-with-words-and-char-n-grams are None
finish running till 58387
labels at url https://www.kaggle.com/code/tunguz/lstm-gru-why-not-both are None
finish running till 58388
labels a

labels at url https://www.kaggle.com/code/tunguz/porto-seguro-gpu-shap-test-1 are ['Porto Seguro’s Safe Driver Prediction']
finish running till 58452
labels at url https://www.kaggle.com/code/tunguz/porto-seguro-gpu-shap-test-2 are ['Porto Seguro’s Safe Driver Prediction']
finish running till 58453
labels at url https://www.kaggle.com/code/tunguz/porto-seguro-gpu-shap-test-3 are ['Porto Seguro’s Safe Driver Prediction']
finish running till 58454
labels at url https://www.kaggle.com/code/tunguz/porto-seguro-with-h2o-automl are ['Porto Seguro’s Safe Driver Prediction']
finish running till 58455
labels at url https://www.kaggle.com/code/tunguz/pubmed-2019-starter-notebook are ['PUBMED title abstracts 2019 baseline']
finish running till 58456
labels at url https://www.kaggle.com/code/tunguz/pulitzer-eda-starter are ['Pulitzer Circulation Data']
finish running till 58457
labels at url https://www.kaggle.com/code/tunguz/quest-logistic-regression-with-word-char-ngrams are ['Google QUEST Q&amp

labels at url https://www.kaggle.com/code/tunguz/tf-embeddings-processed-train-files-first-half are ['Toxic Comment Classification Challenge']
finish running till 58506
labels at url https://www.kaggle.com/code/tunguz/tf-embeddings-processed-train-files-second-half are ['Toxic Comment Classification Challenge']
finish running till 58507
labels at url https://www.kaggle.com/code/tunguz/the-one-table are ['TalkingData Mobile User Demographics']
finish running till 58508
labels at url https://www.kaggle.com/code/tunguz/threat-with-h2o-automl are ['Toxic Comment Classification Challenge', 'TF Embedding Files Joiner']
finish running till 58509
labels at url https://www.kaggle.com/code/tunguz/time-series-forecast-example-with-prophet are ['Web Traffic Time Series Forecasting']
finish running till 58510
labels at url https://www.kaggle.com/code/tunguz/titanic-with-rapids-ds-on-gpus are ['RAPIDS', 'Titanic - Machine Learning from Disaster']
finish running till 58511
labels at url https://www.k

...

labels at url https://www.kaggle.com/code/zakariajoudar/chatgpt-chatbot are ['Regression with a Tabular Paris Housing Price Dataset']
finish running till 63314
labels at url https://www.kaggle.com/code/zakariajoudar/chatgpt-simulation-2023 are ['No attached data sources']
finish running till 63315
labels at url https://www.kaggle.com/code/zakariajoudar/cnn-digit-recognizer are ['Digit Recognizer']
finish running till 63316
labels at url https://www.kaggle.com/code/zakariajoudar/gradient-boosting-smoke-detection are ['Smoke Detection Dataset ']
finish running till 63317
labels at url https://www.kaggle.com/code/zakariajoudar/lstm-learning-equality are ['Fake and real news dataset', 'Learning Equality - Curriculum Recommendations']
finish running till 63318
labels at url https://www.kaggle.com/code/zakariajoudar/mask-car-bicycle are ['Camvid Tiramisu']
finish running till 63319
labels at url https://www.kaggle.com/code/zakariajoudar/pydicom-breast-cancer are ['RSNA Screening Mammography 

labels at url https://www.kaggle.com/code/zeus75/quora-with-bilstm are ['Quora Insincere Questions Classification']
finish running till 63371
labels at url https://www.kaggle.com/code/zeus75/santander-with-lgbm are None
finish running till 63372
labels at url https://www.kaggle.com/code/zeus75/sql-scavenger-hunt-day-1-rf75 are None
finish running till 63373
labels at url https://www.kaggle.com/code/zeus75/sql-scavenger-hunt-day-2 are None
finish running till 63374
labels at url https://www.kaggle.com/code/zeus75/sql-scavenger-hunt-day-3 are None
finish running till 63375
labels at url https://www.kaggle.com/code/zeus75/sql-scavenger-hunt-day-4 are None
finish running till 63376
labels at url https://www.kaggle.com/code/zeus75/sql-scavenger-hunt-day-5 are None
finish running till 63377
labels at url https://www.kaggle.com/code/zeus75/u-net-using-keras are None
finish running till 63378
labels at url https://www.kaggle.com/code/zeus75/welcome-to-data-science-in-r-workbook are None
finish

labels at url https://www.kaggle.com/code/ziadhamadafathy/sentiment-analysis-for-books-reviews are None
finish running till 63437
labels at url https://www.kaggle.com/code/ziadhamadafathy/sign-language-classification are None
finish running till 63438
labels at url https://www.kaggle.com/code/ziadhamadafathy/some-analysis-for-data-science-job-salaries are None
finish running till 63439
labels at url https://www.kaggle.com/code/ziadhamadafathy/using-some-classifiers-models-of-ml are None
finish running till 63440
labels at url https://www.kaggle.com/code/ziadhamadafathy/using-some-models-in-classification-accuracy-95 are ['Data Science London + Scikit-learn']
finish running till 63441
labels at url https://www.kaggle.com/code/ziadhamadafathy/visualization-and-predict-heart-attack-acc-88 are ['Heart Attack Analysis &amp; Prediction Dataset']
finish running till 63442
labels at url https://www.kaggle.com/code/ziadhamadafathy/visualizing-and-predict-heart-disease-acc-98-5 are ['Indicators 

labels at url https://www.kaggle.com/code/zohaib123/image-caption-generator-using-vgg16-bigru are ['Flickr 8k Dataset']
finish running till 63489
labels at url https://www.kaggle.com/code/zohaib123/model-development-a-to-z-with-q-ans are ['No attached data sources']
finish running till 63490
labels at url https://www.kaggle.com/code/zohaib123/notebook423435995a are ['Titanic - Machine Learning from Disaster']
finish running till 63491
labels at url https://www.kaggle.com/code/zohaib123/perform-eda-beginner-s-guide-titanic are ['Titanic - Machine Learning from Disaster']
finish running till 63492
labels at url https://www.kaggle.com/code/zohaib123/predict-patient-class-decision-tree are ['No attached data sources']
finish running till 63493
labels at url https://www.kaggle.com/code/zohaib123/regression-how-best-bed-fit-models-working are ['No attached data sources']
finish running till 63494
labels at url https://www.kaggle.com/code/zohaib123/telecusts-prediction-k-nearest-neighbors are

labels at url https://www.kaggle.com/code/zzettrkalpakbal/cart-analysis-for-stroke-prediction-acc-94-9 are ['Brain stroke prediction dataset']
finish running till 63546
labels at url https://www.kaggle.com/code/zzettrkalpakbal/catboost-classification-optuna-smote-rfecv are ['Biomechanical features of orthopedic patients']
finish running till 63547
labels at url https://www.kaggle.com/code/zzettrkalpakbal/catboost-shap-re-sampling-methods are ['Biomechanical features of orthopedic patients']
finish running till 63548
labels at url https://www.kaggle.com/code/zzettrkalpakbal/comma-separated-csv-but-feature-type-object are ['Fertile man semen parameters 2020 – WHO']
finish running till 63549
labels at url https://www.kaggle.com/code/zzettrkalpakbal/covid-booster-shot are ['[Private Datasource]']
finish running till 63550
labels at url https://www.kaggle.com/code/zzettrkalpakbal/covid-data-leakage are ['Coronavirus-Dataset France', 'Tabular Playground Series - Sep 2022']
finish running til

In [14]:
users_code_location_df.to_csv(f'users_code_location.csv')